# Domain Fine-Tuning for Supply Chain Sustainability Q&A

**EcoSupplyAI** - Fine-Tuning Pipeline

---

## Goal

Fine-tune a small language model on supply chain sustainability domain data to significantly
improve answer quality for EcoSupplyAI's core use cases. By training on curated
instruction-response pairs covering ESG scoring, emissions analysis, regulatory compliance,
supplier risk assessment, and supply chain optimization, the fine-tuned model will produce
more accurate, domain-specific responses compared to the base pre-trained model.

### Why Fine-Tune?

- **Domain specificity**: General-purpose LLMs lack deep knowledge of supply chain sustainability metrics, ESG frameworks, and industry-specific regulations.
- **Consistency**: Fine-tuned models produce more consistent, structured outputs aligned with EcoSupplyAI's reporting formats.
- **Efficiency**: A smaller fine-tuned model can outperform a larger general model on domain tasks, reducing inference costs.
- **Data privacy**: Running a fine-tuned model on-premises or in a private cloud keeps sensitive supplier data secure.

### Approach

We use **LoRA (Low-Rank Adaptation)** for parameter-efficient fine-tuning, which:
- Trains only a small fraction of model parameters (~1-2%)
- Reduces GPU memory requirements dramatically
- Enables fine-tuning on consumer-grade hardware
- Produces modular adapters that can be swapped or merged

In [ ]:
# Core imports for fine-tuning pipeline
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

# Check device availability
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Dataset Preparation

We create an instruction-tuning dataset specifically tailored for supply chain sustainability
use cases. Each sample follows the **Alpaca instruction format**:

```json
{
  "instruction": "The task or question to answer",
  "input": "Optional additional context or data",
  "output": "The expected model response"
}
```

The dataset covers five core domains:
1. **ESG Scoring** - Environmental, Social, and Governance metrics evaluation
2. **Supply Chain Regulations** - Compliance with CSRD, CBAM, EUDR, and other frameworks
3. **Emission Analysis** - Scope 1/2/3 greenhouse gas accounting
4. **Supplier Comparison** - Multi-criteria supplier evaluation and ranking
5. **Risk Assessment** - Supply chain risk identification and mitigation

In [ ]:
# Synthetic training dataset for supply chain sustainability domain
training_data = [
    # --- ESG Scoring Questions ---
    {
        "instruction": "Calculate the ESG score for a supplier based on the provided metrics.",
        "input": "Environmental: Carbon emissions 450 tCO2e, water usage 12000 m3, renewable energy 35%. Social: Worker safety incidents 2, fair wage compliance 92%, diversity ratio 0.41. Governance: Board independence 60%, anti-corruption policy yes, audit frequency quarterly.",
        "output": "ESG Score Breakdown:\n\n**Environmental (Weight: 40%)**: Score 62/100\n- Carbon emissions (450 tCO2e): Moderate, scores 55/100. Industry benchmark is <300 tCO2e for top quartile.\n- Water usage (12,000 m3): Above average consumption, scores 50/100.\n- Renewable energy (35%): Below the 50% target, scores 70/100.\nWeighted Environmental Score: 24.8/40\n\n**Social (Weight: 30%)**: Score 74/100\n- Worker safety (2 incidents): Acceptable but improvement needed, scores 70/100.\n- Fair wage compliance (92%): Strong, scores 85/100. Target is 95%+.\n- Diversity ratio (0.41): Near parity, scores 78/100.\nWeighted Social Score: 22.2/30\n\n**Governance (Weight: 30%)**: Score 80/100\n- Board independence (60%): Meets minimum threshold, scores 75/100.\n- Anti-corruption policy: Present, scores 85/100.\n- Quarterly audits: Best practice frequency, scores 90/100.\nWeighted Governance Score: 24.0/30\n\n**Overall ESG Score: 71.0/100** (Rating: B+)\n\nKey recommendations: Prioritize reducing carbon emissions below 350 tCO2e and increasing renewable energy share to 50%+."
    },
    {
        "instruction": "What are the key components of an ESG scoring framework for supply chain evaluation?",
        "input": "",
        "output": "An ESG scoring framework for supply chain evaluation consists of three pillars:\n\n**1. Environmental (E)**:\n- Greenhouse gas emissions (Scope 1, 2, and 3)\n- Energy consumption and renewable energy percentage\n- Water usage and waste management\n- Biodiversity impact and land use\n- Circular economy practices and recycling rates\n\n**2. Social (S)**:\n- Labor practices and fair wage compliance\n- Workplace health and safety incident rates\n- Diversity, equity, and inclusion metrics\n- Community engagement and social impact\n- Human rights due diligence in supply chain\n\n**3. Governance (G)**:\n- Board composition and independence\n- Anti-corruption and anti-bribery policies\n- Transparency and reporting frequency\n- Risk management frameworks\n- Stakeholder engagement practices\n\n**Scoring Methodology**:\n- Each metric is scored on a 0-100 scale against industry benchmarks\n- Pillar weights are typically 40% E, 30% S, 30% G (adjustable by industry)\n- Final score maps to ratings: A+ (90-100), A (80-89), B+ (70-79), B (60-69), C (50-59), D (<50)\n- Scores should be updated quarterly with annual deep assessments"
    },
    {
        "instruction": "Compare the ESG performance of two suppliers and recommend which one to prioritize.",
        "input": "Supplier A: ESG score 78, carbon intensity 0.45 kgCO2/unit, 100% audit compliance, lead time 14 days, cost per unit $12.50. Supplier B: ESG score 65, carbon intensity 0.82 kgCO2/unit, 85% audit compliance, lead time 8 days, cost per unit $10.20.",
        "output": "**Supplier Comparison Analysis**:\n\n| Metric | Supplier A | Supplier B | Advantage |\n|--------|-----------|-----------|-----------|\n| ESG Score | 78 (B+) | 65 (B) | Supplier A (+13) |\n| Carbon Intensity | 0.45 kgCO2/unit | 0.82 kgCO2/unit | Supplier A (-45%) |\n| Audit Compliance | 100% | 85% | Supplier A |\n| Lead Time | 14 days | 8 days | Supplier B (-6 days) |\n| Cost per Unit | $12.50 | $10.20 | Supplier B (-$2.30) |\n\n**Recommendation: Supplier A** (with conditions)\n\nRationale:\n1. Superior ESG score (78 vs 65) aligns with sustainability commitments and reduces regulatory risk.\n2. The 45% lower carbon intensity directly reduces Scope 3 emissions.\n3. 100% audit compliance indicates operational reliability.\n4. The $2.30/unit cost premium is offset by avoided carbon costs.\n\nAction items:\n- Negotiate volume-based pricing with Supplier A to close the cost gap.\n- Develop Supplier B improvement plan if dual-sourcing is needed.\n- Set quarterly ESG review cadence for both suppliers."
    },
    {
        "instruction": "Interpret this supplier's ESG trend data and identify areas of concern.",
        "input": "Q1: E=72, S=68, G=75. Q2: E=70, S=71, G=74. Q3: E=65, S=73, G=72. Q4: E=58, S=74, G=70.",
        "output": "**ESG Trend Analysis**:\n\n**Environmental Score: DECLINING (72 -> 58, -19.4%)**\n- Consistent quarter-over-quarter decline of 4-7 points.\n- This is the most critical concern. The drop from 72 to 58 crosses the B-to-C threshold.\n- **Action Required**: Immediate investigation and corrective action plan within 30 days.\n\n**Social Score: IMPROVING (68 -> 74, +8.8%)**\n- Steady improvement of 1-3 points per quarter.\n- Continue current initiatives; target 80+ by next year.\n\n**Governance Score: SLIGHTLY DECLINING (75 -> 70, -6.7%)**\n- Gradual erosion warrants monitoring.\n- Recommend governance review in next quarterly assessment.\n\n**Overall Assessment**: The environmental decline is alarming and should trigger the supplier risk escalation protocol. Recommend placing this supplier on the ESG watchlist."
    },
    # --- Supply Chain Regulation Questions ---
    {
        "instruction": "Explain how the EU Corporate Sustainability Reporting Directive (CSRD) impacts supply chain management.",
        "input": "",
        "output": "**CSRD Impact on Supply Chain Management**:\n\nThe EU CSRD, effective from January 2024, fundamentally transforms supply chain transparency:\n\n**1. Scope**: Applies to ~50,000 companies including non-EU companies with significant EU revenue (>150M EUR). Covers the entire value chain.\n\n**2. Key Requirements**:\n- Double materiality assessment\n- Mandatory Scope 3 emissions reporting\n- Supply chain human rights and environmental due diligence\n- Biodiversity impact reporting\n\n**3. Procurement Implications**:\n- Suppliers must provide verified ESG data or face delisting\n- Contracts need sustainability KPI clauses\n- Annual sustainability assessments mandatory for Tier 1 suppliers\n\n**4. Timeline**: 2024 for large entities, 2025 for large companies, 2026 for listed SMEs.\n\n**5. Penalties**: Fines, director liability, and exclusion from public procurement."
    },
    {
        "instruction": "What is the Carbon Border Adjustment Mechanism (CBAM) and how should importers prepare?",
        "input": "",
        "output": "**Carbon Border Adjustment Mechanism (CBAM)**:\n\n**Overview**: CBAM is the EU's carbon tariff on carbon-intensive imports to prevent carbon leakage.\n\n**Covered Sectors**: Iron/steel, cement, aluminium, fertilizers, electricity, hydrogen.\n\n**Timeline**:\n- Oct 2023 - Dec 2025: Transitional period (reporting only)\n- Jan 2026+: Full implementation with CBAM certificates\n\n**Preparation Steps**:\n1. Map supply chain emissions for covered sectors\n2. Calculate CBAM liability using embedded emissions data\n3. Engage suppliers on verified emissions data\n4. Register as authorized CBAM declarant\n5. Evaluate total cost including CBAM surcharges and consider supplier switching"
    },
    {
        "instruction": "Summarize the EU Deforestation Regulation (EUDR) requirements for supply chain compliance.",
        "input": "Our company imports palm oil, soy, and timber products into the EU market.",
        "output": "**EUDR Compliance for Palm Oil, Soy, and Timber**:\n\nThe EUDR prohibits placing products linked to deforestation after December 31, 2020 on the EU market.\n\n**Key Steps**:\n1. **Geolocation Mapping**: GPS coordinates of all production plots\n2. **Due Diligence System**: Risk assessment using satellite monitoring\n3. **Traceability**: Full chain of custody from production to import\n4. **Due Diligence Statement**: Electronic submission before placing goods on market\n5. **Timeline**: Large operators by Dec 30, 2024; SMEs by June 30, 2025\n\n**Risk Mitigation**: Engage suppliers on geolocation data, invest in satellite monitoring, consider certified low-risk regions."
    },
    {
        "instruction": "What supply chain due diligence obligations exist under the German Supply Chain Act (LkSG)?",
        "input": "",
        "output": "**German Supply Chain Act (LkSG) Obligations**:\n\n**Applicability**: Companies with 1,000+ employees in Germany (since Jan 2024).\n\n**Core Obligations**:\n1. Risk management system for human rights and environmental risks\n2. Annual risk analysis of Tier 1 suppliers\n3. Preventive measures: supplier code of conduct, training, audits\n4. Corrective actions for violations\n5. Accessible grievance mechanism\n6. Annual public reporting to BAFA\n\n**Penalties**: Up to 2% of global revenue, exclusion from public procurement for 3 years."
    },
    # --- Emission Analysis Questions ---
    {
        "instruction": "Analyze the Scope 3 emissions breakdown for a manufacturing company and suggest reduction strategies.",
        "input": "Scope 3 categories: Purchased goods (45%), Transportation upstream (18%), Business travel (5%), Employee commuting (3%), Waste (2%), Transportation downstream (12%), Use of sold products (10%), End-of-life treatment (5%). Total Scope 3: 125,000 tCO2e.",
        "output": "**Scope 3 Emissions Analysis** (Total: 125,000 tCO2e):\n\n**Priority 1 - Purchased Goods (56,250 tCO2e, 45%)**:\n- Supplier engagement: Set SBTs for top 20 suppliers\n- Material substitution to low-carbon alternatives\n- Target: -25% by 2027 (save ~14,000 tCO2e)\n\n**Priority 2 - Upstream Transportation (22,500 tCO2e, 18%)**:\n- Modal shift: 30% from road to rail/sea\n- AI-driven route optimization\n- Target: -20% by 2027 (save ~4,500 tCO2e)\n\n**Priority 3 - Downstream Transportation (15,000 tCO2e, 12%)**:\n- Shipment consolidation and last-mile optimization\n- Partner with carriers on fleet electrification\n- Target: -15% by 2027 (save ~2,250 tCO2e)\n\n**Combined Target**: -20,750 tCO2e (-16.6%) by 2027."
    },
    {
        "instruction": "Explain how to calculate Scope 3 Category 1 (Purchased Goods and Services) emissions.",
        "input": "",
        "output": "**Calculating Scope 3 Category 1**:\n\n**Three Methods** (most to least accurate):\n\n**1. Supplier-Specific**: Collect actual emissions data from suppliers. Highest accuracy.\n\n**2. Hybrid** (Recommended): Use supplier data where available, fill gaps with industry emission factors from DEFRA/ecoinvent/GaBi.\n\n**3. Spend-Based**: Use financial spend x EEIO emission factors. Lowest accuracy but good for screening.\n\n**Process**:\n1. Categorize all purchases\n2. Collect activity data (tonnes, kWh, units, or spend)\n3. Select emission factors\n4. Calculate: Activity Data x Emission Factor = tCO2e\n5. Sum all categories\n6. Document methodology and assumptions\n\n**Best Practice**: Start spend-based, then progressively improve top categories to supplier-specific data."
    },
    {
        "instruction": "Convert this energy consumption data to carbon emissions using appropriate emission factors.",
        "input": "Electricity: 2,500,000 kWh (grid mix, Germany). Natural gas: 850,000 kWh. Diesel fleet: 120,000 liters. Refrigerant leakage: 15 kg R-410A.",
        "output": "**Carbon Emissions Calculation**:\n\n| Source | Activity Data | Emission Factor | tCO2e | Scope |\n|--------|--------------|-----------------|-------|-------|\n| Grid Electricity (DE) | 2,500,000 kWh | 0.366 kgCO2e/kWh | 915.0 | Scope 2 |\n| Natural Gas | 850,000 kWh | 0.202 kgCO2e/kWh | 171.7 | Scope 1 |\n| Diesel Fleet | 120,000 liters | 2.68 kgCO2e/L | 321.6 | Scope 1 |\n| R-410A Refrigerant | 15 kg | 2,088 kgCO2e/kg | 31.3 | Scope 1 |\n\n**Summary**: Scope 1: 524.6 tCO2e (36.3%) | Scope 2: 915.0 tCO2e (63.7%)\n**Total: 1,439.6 tCO2e**\n\n**Reduction Opportunities**: Renewable electricity tariff (-915 tCO2e), fleet electrification (-321.6 tCO2e), low-GWP refrigerant (-90% refrigerant emissions)."
    },
    {
        "instruction": "What is the difference between location-based and market-based Scope 2 emissions reporting?",
        "input": "",
        "output": "**Location-Based vs Market-Based Scope 2**:\n\n**Location-Based**: Uses average grid emission factors for the region. Reflects physical grid mix. Unaffected by RECs/certificates.\n\n**Market-Based**: Uses emission factors from contractual instruments (PPAs, RECs, GOs). Reflects purchasing decisions. Can be zero with 100% renewable procurement.\n\n| Aspect | Location-Based | Market-Based |\n|--------|---------------|-------------|\n| Data source | Grid average factors | Contractual instruments |\n| Reflects | Physical reality | Purchasing choices |\n| RECs impact | No effect | Reduces to zero |\n\n**GHG Protocol**: Companies must report BOTH methods. SBTi and CDP accept market-based for targets but require location-based disclosure.\n\n**Best Practice**: Report both, set targets using market-based, prefer PPAs over unbundled RECs."
    },
    # --- Supplier Comparison Questions ---
    {
        "instruction": "Rank these suppliers using a weighted multi-criteria sustainability assessment.",
        "input": "Criteria weights: ESG Score (30%), Carbon Intensity (25%), Cost (20%), Reliability (15%), Innovation (10%). Supplier X: ESG=82, Carbon=0.35, Cost=$11, Reliability=95%, Innovation=7/10. Supplier Y: ESG=71, Carbon=0.52, Cost=$9.50, Reliability=98%, Innovation=8/10. Supplier Z: ESG=90, Carbon=0.28, Cost=$13.50, Reliability=88%, Innovation=9/10.",
        "output": "**Multi-Criteria Sustainability Assessment**:\n\n**Normalized and Weighted Scores**:\n\n| Metric (Weight) | Supplier X | Supplier Y | Supplier Z |\n|----------------|-----------|-----------|-----------|\n| ESG (30%) | 24.6 | 21.3 | 27.0 |\n| Carbon (25%) | 21.9 | 14.4 | 25.0 |\n| Cost (20%) | 16.3 | 20.0 | 11.1 |\n| Reliability (15%) | 14.5 | 15.0 | 13.5 |\n| Innovation (10%) | 7.0 | 8.0 | 9.0 |\n\n**Final Ranking**:\n1. Supplier Z: 85.6 (Preferred)\n2. Supplier X: 84.3 (Approved)\n3. Supplier Y: 78.7 (Conditional)\n\n**Recommendation**: Z as primary, X as secondary. Engage Y on carbon reduction for cost optimization backup."
    },
    {
        "instruction": "Evaluate a supplier's sustainability improvement plan and provide feedback.",
        "input": "Supplier proposes: 1) Install solar panels (500kW) by Q3 2025, 2) Switch to electric forklifts by 2026, 3) Implement ISO 14001 by Q4 2025, 4) Reduce water usage by 15% through recycling, 5) Achieve zero waste to landfill by 2027.",
        "output": "**Sustainability Plan Evaluation: B+ (Good, with gaps)**\n\n1. **Solar (500kW) by Q3 2025** - 9/10: Strong initiative, ~220 tCO2e/year reduction. Add battery storage consideration.\n\n2. **Electric Forklifts by 2026** - 7/10: Good but needs quarterly milestones and charging infrastructure plan.\n\n3. **ISO 14001 by Q4 2025** - 8/10: Essential. Ensure gap analysis is underway. Consider ISO 50001 too.\n\n4. **Water -15% via Recycling** - 6/10: Needs baseline, quarterly targets, specific technology. Industry leaders achieve 30-40%.\n\n5. **Zero Waste to Landfill by 2027** - 7/10: Define zero (<1%), include waste audit and interim targets.\n\n**Missing**: No Scope 3 targets, no social goals, no governance improvements. Add SBTi alignment."
    },
    # --- Risk Assessment Questions ---
    {
        "instruction": "Conduct a sustainability risk assessment for a supply chain sourcing from Southeast Asia.",
        "input": "Industry: Electronics manufacturing. Key materials: Rare earth elements, lithium, cobalt, copper, plastics. Main sourcing countries: Vietnam, Thailand, Indonesia, Philippines.",
        "output": "**SE Asia Electronics Supply Chain Risk Assessment**:\n\n| Risk | Likelihood | Impact | Level |\n|------|-----------|--------|-------|\n| Deforestation/Land use | High | High | Critical |\n| Labor rights violations | Med-High | High | High |\n| Water stress | Medium | High | High |\n| Climate physical risks | High | Medium | High |\n| Conflict minerals | Medium | High | High |\n\n**Key Risks**: Mining-driven deforestation (EUDR risk), forced labor in cobalt chains, water scarcity in Thailand/Vietnam, typhoon/flood disruption (30-40% annual probability), conflict mineral regulations.\n\n**Actions**: Tiered due diligence, on-site Tier 1 assessments within 6 months, satellite monitoring, multi-sourcing strategy, RMAP-certified smelters."
    },
    {
        "instruction": "Assess the financial risk of not complying with upcoming sustainability regulations.",
        "input": "Company revenue: 500M EUR. Operating in EU market. Currently non-compliant with CSRD and CBAM requirements. 40% of materials sourced from high-carbon suppliers.",
        "output": "**Financial Risk: Regulatory Non-Compliance** (Revenue: 500M EUR)\n\n**CSRD Risk**: Fines up to 25M EUR (5% turnover), director liability, public procurement exclusion (50-75M EUR contract risk). Total: 25-100M EUR/year.\n\n**CBAM Risk**: 200M EUR affected materials, ~100K tCO2e embedded, ~8.5M EUR annual cost at 85 EUR/tCO2e. Penalties: 3x certificate price. Total: 8.5-25.5M EUR/year.\n\n**Market Risk**: 15-20% revenue at risk from customer ESG requirements (75-100M EUR).\n\n**Cost Comparison**:\n| Scenario | 3-Year Cumulative |\n|----------|-------------------|\n| Non-compliance | 200M EUR |\n| Compliance investment | 33M EUR |\n| **Net savings** | **167M EUR** |\n\nROI of compliance: 5:1 over three years."
    },
    {
        "instruction": "Identify the top climate-related physical risks for a global supply chain and recommend mitigation strategies.",
        "input": "Supply chain spans: Factories in coastal Vietnam and Bangladesh, warehouses in Netherlands and Houston TX, shipping routes through Suez Canal and South China Sea.",
        "output": "**Climate Physical Risk Assessment**:\n\n1. **Coastal Vietnam** (VERY HIGH): Typhoons, sea level rise, Mekong flooding. Mitigation: Inland backup capacity, 30-day safety stock, facility hardening.\n\n2. **Bangladesh** (VERY HIGH): Cyclones, monsoon floods, heat stress. Mitigation: Dual-sourcing, climate-resilient buildings, heat protocols.\n\n3. **Houston TX** (HIGH): Hurricanes (Harvey precedent). Mitigation: Inland backup, business continuity plan.\n\n4. **South China Sea** (HIGH): Typhoon season, geopolitical risks. Mitigation: Seasonal buffers, Pacific routing alternatives.\n\n5. **Netherlands** (MEDIUM): Long-term sea level/river flooding. Mitigation: Verify Delta Plan protection, flood insurance.\n\n6. **Suez Canal** (MEDIUM): Geopolitical/drought risk. Mitigation: Cape of Good Hope contingency.\n\n**Portfolio Actions**: TCFD climate scenario analysis, supplier resilience scorecards, 2-3% budget for resilience investments."
    },
    {
        "instruction": "How should a company set Science-Based Targets for supply chain emissions reduction?",
        "input": "",
        "output": "**Setting Science-Based Targets (SBTs)**:\n\n**Step 1: Commit** - Submit letter to SBTi (24 months to develop targets).\n\n**Step 2: Emissions Inventory** - Complete Scopes 1, 2, 3. Scope 3 target required if >40% of total.\n\n**Step 3: Set Targets**:\n- Near-term (5-10yr): Scope 1+2 min 4.2%/year; Scope 3 min 2.5%/year\n- Long-term (by 2050): 90% reduction, 10% via carbon removal\n\n**Step 4: Supply Chain Actions**:\n- Supplier engagement: 67% of Scope 3 suppliers set own SBTs within 5 years\n- Physical intensity and absolute targets\n\n**Step 5: Submit and Validate** (6-8 months)\n\n**Step 6: Track via CDP**, recalculate base year as needed, review every 5 years.\n\n**Pitfalls**: Underestimating Scope 3 data effort, late supplier engagement, no decarbonization roadmap."
    },
    {
        "instruction": "Generate a sustainability risk score for a new supplier onboarding request.",
        "input": "Supplier: ChemCorp Ltd. Industry: Chemical manufacturing. Location: Gujarat, India. Employees: 450. Certifications: ISO 9001 (yes), ISO 14001 (no), SA8000 (no). Annual revenue: $28M. Products: Industrial solvents, adhesives. Environmental incidents last 3 years: 2. Labor disputes: 1.",
        "output": "**Supplier Risk Scoring: ChemCorp Ltd. - Score: 62/100 (MEDIUM-HIGH RISK)**\n\n| Factor | Weight | Score | Weighted |\n|--------|--------|-------|----------|\n| Industry Risk | 15% | 35 | 5.3 |\n| Geographic Risk | 15% | 45 | 6.8 |\n| Environmental Mgmt | 20% | 40 | 8.0 |\n| Social/Labor | 15% | 50 | 7.5 |\n| Governance | 10% | 55 | 5.5 |\n| Size/Capacity | 10% | 65 | 6.5 |\n| Financial Stability | 15% | 70 | 10.5 |\n\n**Red Flags**: No ISO 14001 in chemical industry, 2 environmental incidents.\n**Amber Flags**: No SA8000, regulatory restriction risk on solvents.\n\n**Decision: CONDITIONAL APPROVAL**\nRequired: Site visit (60 days), ISO 14001 commitment (18 months), labor audit (90 days), incident root cause review. Contract must include quarterly ESG KPIs and audit rights."
    },
    {
        "instruction": "What metrics should be tracked in a sustainable procurement dashboard?",
        "input": "",
        "output": "**Sustainable Procurement Dashboard Metrics**:\n\n**Environmental**: Scope 3 Cat 1 emissions, supplier carbon intensity, % suppliers with SBTs, renewable energy rate, water consumption, waste diversion rate.\n\n**Social**: Labor audit pass rate, human rights incidents, living wage compliance, supplier diversity spend, H&S incident rate.\n\n**Governance**: ESG assessment completion rate, avg supplier ESG score/trend, code of conduct acknowledgment, sustainability clause inclusion in contracts.\n\n**Performance**: Sustainable spend %, suppliers on improvement plans, ESG score improvement YoY, sustainable vs conventional cost tracking.\n\n**Risk**: High-risk supplier count, geographic concentration index, regulatory readiness (CSRD/CBAM/EUDR), ESG-linked disruption events.\n\n**Design**: RAG status executive view, drill-down by category/region/supplier, quarterly trend lines, peer benchmarks, automated threshold alerts."
    },
    {
        "instruction": "Explain the concept of double materiality in the context of supply chain sustainability.",
        "input": "",
        "output": "**Double Materiality in Supply Chain Sustainability**:\n\n**Financial Materiality (Outside-In)**: How sustainability issues affect the company financially. Examples: climate disruptions causing revenue loss, carbon regulations increasing costs.\n\n**Impact Materiality (Inside-Out)**: How the company affects people and environment. Examples: supplier pollution harming communities, deforestation in supply chain.\n\n**Practical Application**:\n| Issue | Financial | Impact |\n|-------|----------|--------|\n| Supplier emissions | CBAM costs | Climate contribution |\n| Child labor | Reputational damage | Human rights harm |\n| Water-intensive sourcing | Disruption risk | Local depletion |\n\n**Under CSRD**: Companies must report BOTH dimensions for their entire value chain.\n\n**Process**: Identify topics, assess both perspectives, engage stakeholders, prioritize material topics from either/both views, report with metrics and targets."
    },
    {
        "instruction": "Draft a supplier sustainability questionnaire for annual ESG assessment.",
        "input": "Focus areas: carbon emissions, water management, labor practices, and governance.",
        "output": "**Annual Supplier ESG Assessment Questionnaire**:\n\n**Section A: Carbon & Energy (25 pts)**\nA1. Do you measure GHG emissions? A2. Annual Scope 1+2? A3. Reduction targets? A4. Renewable energy %? A5. Science-Based Targets status? A6. CDP reporting? A7. Top 3 efficiency initiatives?\n\n**Section B: Water Management (20 pts)**\nB1. Measure withdrawal by source? B2. Annual consumption (m3)? B3. Operations in water-stressed areas? B4. Recycling/reuse %? B5. Reduction targets? B6. Water incidents in 3 years?\n\n**Section C: Labor & Human Rights (30 pts)**\nC1. Human rights policy? C2. Safety incident rate? C3. Third-party labor audits? C4. Living wage compliance? C5. Grievance mechanism? C6. Working hours/overtime policy? C7. Supply chain due diligence? C8. Workforce diversity?\n\n**Section D: Governance (25 pts)**\nD1. Certifications (ISO 14001, SA8000)? D2. Board sustainability oversight? D3. Anti-corruption policy? D4. Annual sustainability report? D5. Supplier code of conduct? D6. Risk management processes?\n\n**Scoring**: A+ (90-100), A (80-89), B+ (70-79), B (60-69), C (50-59), Fail (<50). Deadline: 30 days."
    }
]

# Save training data to JSON file
output_path = "fine_tune_data.json"
with open(output_path, "w") as f:
    json.dump(training_data, f, indent=2)

print(f"Created training dataset with {len(training_data)} examples")
print(f"Saved to: {output_path}")

# Display dataset statistics
df = pd.DataFrame(training_data)
df["instruction_length"] = df["instruction"].str.len()
df["input_length"] = df["input"].str.len()
df["output_length"] = df["output"].str.len()
df["total_length"] = df["instruction_length"] + df["input_length"] + df["output_length"]

print(f"\nDataset Statistics:")
print(f"  Total examples: {len(df)}")
print(f"  Avg instruction length: {df['instruction_length'].mean():.0f} chars")
print(f"  Avg output length: {df['output_length'].mean():.0f} chars")
print(f"  Avg total length: {df['total_length'].mean():.0f} chars")
print(f"  Examples with input context: {(df['input_length'] > 0).sum()}")
print(f"  Examples without input: {(df['input_length'] == 0).sum()}")

## 2. Data Preprocessing

We convert our instruction/input/output format into a single text sequence for the
language model using the Alpaca-style prompt template.

**Tokenization Strategy**:
1. Format with `### Instruction:`, `### Input:` (optional), `### Response:` delimiters
2. Tokenize with padding to uniform length and truncation to max context window
3. Set labels equal to input_ids for causal language modeling
4. Configure proper padding token (many causal LM tokenizers lack one by default)

In [ ]:
# Model selection - using TinyLlama for efficient fine-tuning
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LENGTH = 1024  # Maximum sequence length for tokenization

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ensure padding token is set (required for batched training)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Max model length: {tokenizer.model_max_length:,}")
print(f"Pad token: '{tokenizer.pad_token}' (id: {tokenizer.pad_token_id})")


def format_prompt(example):
    """Convert instruction/input/output dict into Alpaca-style prompt."""
    if example["input"].strip():
        prompt = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    else:
        prompt = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return prompt


def tokenize_function(examples):
    """Tokenize formatted prompts for causal language modeling."""
    texts = [format_prompt({
        "instruction": inst,
        "input": inp,
        "output": out
    }) for inst, inp, out in zip(
        examples["instruction"],
        examples["input"],
        examples["output"]
    )]

    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )

    # For causal LM, labels = input_ids (model predicts next token)
    tokenized["labels"] = tokenized["input_ids"].clone()

    return tokenized


# Load data and create HuggingFace Dataset
with open("fine_tune_data.json", "r") as f:
    raw_data = json.load(f)

dataset = Dataset.from_list(raw_data)

# Split into train and validation sets (90/10)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]

print(f"\nTrain examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

# Tokenize datasets
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training data",
)
tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation data",
)

print(f"\nTokenized train shape: {tokenized_train.shape}")
print(f"Tokenized val shape: {tokenized_val.shape}")

# Show a sample formatted prompt
sample_prompt = format_prompt(raw_data[0])
print(f"\n{'='*60}")
print("Sample formatted prompt (first 500 chars):")
print(f"{'='*60}")
print(sample_prompt[:500] + "...")

## 3. LoRA Configuration

**LoRA (Low-Rank Adaptation)** is a parameter-efficient fine-tuning technique that:

- Freezes all original model weights
- Injects small trainable low-rank matrices into transformer attention layers
- Typically trains only **1-3%** of the total parameters
- Achieves comparable performance to full fine-tuning at a fraction of the cost

### Key Hyperparameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| `r` | 16 | Rank of the low-rank decomposition |
| `lora_alpha` | 32 | Scaling factor (alpha = 2*r) |
| `lora_dropout` | 0.1 | Dropout on LoRA layers |
| `target_modules` | q_proj, v_proj, k_proj, o_proj | Attention layers to adapt |

In [ ]:
# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True,
)

# Enable gradient checkpointing to reduce memory usage
model.gradient_checkpointing_enable()

print(f"Base model loaded: {MODEL_NAME}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Model dtype: {model.dtype}")

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                          # Rank of low-rank decomposition
    lora_alpha=32,                 # Scaling factor (alpha/r = scaling)
    lora_dropout=0.1,              # Dropout for regularization
    target_modules=[               # Attention layers to adapt
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    bias="none",                   # Don't train bias parameters
    modules_to_save=None,          # No additional modules to fully train
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print parameter efficiency analysis
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable_params / total_params

print(f"\n{'='*60}")
print("LoRA Parameter Efficiency Analysis")
print(f"{'='*60}")
print(f"Total parameters:     {total_params:>15,}")
print(f"Trainable parameters: {trainable_params:>15,}")
print(f"Frozen parameters:    {total_params - trainable_params:>15,}")
print(f"Trainable %:          {trainable_pct:>14.2f}%")
print(f"Memory savings:       ~{(1 - trainable_pct/100)*100:.1f}% less GPU memory needed")
print(f"{'='*60}")

# Print model architecture summary showing LoRA layers
model.print_trainable_parameters()

## 4. Training Configuration

Training settings optimized for small dataset fine-tuning with memory efficiency.

**Important**: Training on CPU is extremely slow. Use a GPU with at least 8GB VRAM
(NVIDIA T4, A10G, or RTX 3070+). The `trainer.train()` call is commented out to
prevent accidental long-running execution.

In [ ]:
# Output directory for model checkpoints and logs
OUTPUT_DIR = "./ecosupplyai-finetuned"

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,         # Effective batch size = 4 * 4 = 16
    learning_rate=2e-4,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),        # Use fp16 only with CUDA
    bf16=False,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    report_to="none",                      # Disable W&B/MLflow for demo
    seed=42,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)

# Data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print(f"Trainer initialized successfully.")
print(f"\nTraining Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size (per device): {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  LR scheduler: {training_args.lr_scheduler_type}")
print(f"  Warmup steps: {training_args.warmup_steps}")
print(f"  FP16: {training_args.fp16}")
print(f"  Output dir: {OUTPUT_DIR}")

# ============================================================
# UNCOMMENT THE FOLLOWING LINE TO START TRAINING
# NOTE: Requires GPU. Training on CPU will be extremely slow.
# Estimated time: ~5-15 minutes on a T4/A10G GPU
# ============================================================

# trainer.train()

print("\n** Training is ready to launch. Uncomment 'trainer.train()' above to begin. **")
print("** GPU with >= 8GB VRAM recommended (NVIDIA T4, A10G, RTX 3070+). **")

## 5. Evaluation

Evaluate the fine-tuned model against the base model:
1. **Qualitative comparison**: Side-by-side responses on test prompts
2. **Perplexity**: Lower perplexity = better domain language understanding
3. **Training loss curve**: Verify convergence without overfitting

In [ ]:
# ============================================================
# Evaluation Functions
# ============================================================

def generate_response(model, tokenizer, prompt, max_new_tokens=512):
    """Generate a response from a model given a prompt."""
    formatted = f"### Instruction:\n{prompt}\n\n### Response:\n"
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Response:" in response:
        response = response.split("### Response:")[-1].strip()
    return response


def calculate_perplexity(model, tokenizer, texts):
    """Calculate perplexity of model on a list of texts."""
    model.eval()
    total_loss = 0
    total_tokens = 0

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                          max_length=MAX_LENGTH).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            total_loss += outputs.loss.item() * inputs["input_ids"].shape[1]
            total_tokens += inputs["input_ids"].shape[1]

    avg_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss)).item()
    return perplexity


# Test prompts for evaluation
test_prompts = [
    "What is a Scope 3 emission and why is it important for supply chains?",
    "How should a company prepare for CBAM compliance?",
    "Evaluate a supplier with an ESG score of 55 and carbon intensity of 1.2 kgCO2/unit.",
    "What are the key risks in sourcing raw materials from Southeast Asia?",
    "Explain the difference between Science-Based Targets near-term and long-term goals.",
]

# ============================================================
# Side-by-side comparison (uncomment after training)
# ============================================================
# base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
# finetuned_model = model  # the trained model
#
# for prompt in test_prompts:
#     print(f"\nPrompt: {prompt}")
#     print(f"\n--- Base Model ---")
#     print(generate_response(base_model, tokenizer, prompt))
#     print(f"\n--- Fine-Tuned Model ---")
#     print(generate_response(finetuned_model, tokenizer, prompt))
#     print("=" * 60)

# ============================================================
# Perplexity comparison (uncomment after training)
# ============================================================
# domain_texts = [format_prompt(ex) for ex in raw_data[-5:]]
# base_ppl = calculate_perplexity(base_model, tokenizer, domain_texts)
# ft_ppl = calculate_perplexity(finetuned_model, tokenizer, domain_texts)
# print(f"Base Perplexity: {base_ppl:.2f}")
# print(f"Fine-Tuned Perplexity: {ft_ppl:.2f}")
# print(f"Improvement: {(1 - ft_ppl/base_ppl)*100:.1f}%")

# ============================================================
# Training Loss Curve (demo with mock data)
# ============================================================
np.random.seed(42)
steps = np.arange(0, 150, 10)
train_loss = 3.2 * np.exp(-0.025 * steps) + 0.8 + np.random.normal(0, 0.08, len(steps))
train_loss = np.maximum(train_loss, 0.75)
val_loss_points = [3.5, 2.1, 1.6]
val_steps = [50, 100, 150]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training Loss
axes[0].plot(steps, train_loss, 'b-', linewidth=1.5, alpha=0.7, label='Training Loss')
axes[0].plot(val_steps, val_loss_points, 'ro-', markersize=8, linewidth=2, label='Validation Loss')
axes[0].set_xlabel('Training Steps', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training & Validation Loss Curve', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 4)

# Plot 2: Perplexity comparison (mock data)
categories = ['ESG\nScoring', 'Emissions\nAnalysis', 'Regulations', 'Supplier\nComparison', 'Risk\nAssessment']
base_ppls = [45.2, 52.8, 48.1, 41.3, 55.6]
ft_ppls = [12.4, 15.1, 13.8, 10.9, 16.2]

x = np.arange(len(categories))
width = 0.35
axes[1].bar(x - width/2, base_ppls, width, label='Base Model', color='#ff6b6b', alpha=0.8)
axes[1].bar(x + width/2, ft_ppls, width, label='Fine-Tuned', color='#51cf66', alpha=0.8)
axes[1].set_xlabel('Domain Category', fontsize=12)
axes[1].set_ylabel('Perplexity (lower is better)', fontsize=12)
axes[1].set_title('Perplexity: Base vs Fine-Tuned Model', fontsize=14)
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('training_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPerplexity Comparison (Demo Data):")
print(f"{'Category':<25} {'Base Model':>12} {'Fine-Tuned':>12} {'Improvement':>12}")
print("-" * 65)
for cat, bp, fp in zip(categories, base_ppls, ft_ppls):
    cat_clean = cat.replace('\n', ' ')
    improvement = (1 - fp/bp) * 100
    print(f"{cat_clean:<25} {bp:>12.1f} {fp:>12.1f} {improvement:>11.1f}%")

## 6. Model Export & Deployment

After training:
1. **Save LoRA adapter** (~10-50MB)
2. **Merge into base model** for standalone deployment
3. **Generate model card** documenting training and limitations

In [ ]:
# ============================================================
# Model Export Pipeline
# ============================================================

ADAPTER_DIR = "./ecosupplyai-lora-adapter"
MERGED_DIR = "./ecosupplyai-merged-model"
EXPORT_DIR = "./ecosupplyai-deployment"


def save_lora_adapter(model, output_dir):
    """Save only the LoRA adapter weights (small, modular)."""
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    print(f"LoRA adapter saved to: {output_dir}")
    adapter_size = sum(
        os.path.getsize(os.path.join(output_dir, f))
        for f in os.listdir(output_dir)
        if os.path.isfile(os.path.join(output_dir, f))
    ) / (1024 * 1024)
    print(f"Adapter size: {adapter_size:.1f} MB")


def merge_and_export(base_model_name, adapter_dir, output_dir):
    """Merge LoRA weights into base model and save complete model."""
    os.makedirs(output_dir, exist_ok=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name, torch_dtype=torch.float16, trust_remote_code=True,
    )
    merged_model = PeftModel.from_pretrained(base_model, adapter_dir)
    merged_model = merged_model.merge_and_unload()
    merged_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Merged model saved to: {output_dir}")
    model_size = sum(
        os.path.getsize(os.path.join(output_dir, f))
        for f in os.listdir(output_dir)
        if os.path.isfile(os.path.join(output_dir, f))
    ) / (1024 * 1024 * 1024)
    print(f"Merged model size: {model_size:.2f} GB")
    return merged_model


def generate_model_card(output_dir, training_config):
    """Generate a model card documenting training and capabilities."""
    model_card = f"""---
language: en
license: apache-2.0
tags: [supply-chain, sustainability, esg, fine-tuned, lora]
pipeline_tag: text-generation
---
# EcoSupplyAI Fine-Tuned Model

Fine-tuned `{training_config['base_model']}` for supply chain sustainability Q&A.
Method: LoRA (r={training_config['lora_r']}, alpha={training_config['lora_alpha']})
Training: {training_config['num_examples']} examples, {training_config['epochs']} epochs, LR {training_config['learning_rate']}
Trainable parameters: ~{training_config['trainable_pct']:.2f}%

## Use Cases
- ESG scoring, emissions analysis, regulatory compliance (CSRD/CBAM/EUDR/LkSG)
- Supplier assessment, risk identification

## Limitations
- Demo dataset; expand to 500-2000+ for production
- Verify regulatory guidance against current legal texts
- ESG scores are indicative; validate with domain experts
"""
    card_path = os.path.join(output_dir, "README.md")
    with open(card_path, "w") as f:
        f.write(model_card)
    print(f"Model card saved to: {card_path}")


# Export configuration
training_config = {
    "base_model": MODEL_NAME,
    "lora_r": 16,
    "lora_alpha": 32,
    "num_examples": len(training_data),
    "epochs": 3,
    "learning_rate": "2e-4",
    "trainable_pct": trainable_pct,
}

# Uncomment after training:
# save_lora_adapter(model, ADAPTER_DIR)
# merged_model = merge_and_export(MODEL_NAME, ADAPTER_DIR, MERGED_DIR)
# generate_model_card(MERGED_DIR, training_config)

print("Export pipeline defined. Uncomment after training completes.")
print(f"  LoRA adapter -> {ADAPTER_DIR}")
print(f"  Merged model -> {MERGED_DIR}")
print(f"  Deployment   -> {EXPORT_DIR}")

## 7. Azure ML Integration

Deploy the fine-tuned model to **Azure Machine Learning** as a managed online endpoint:

- Auto-scaling, managed GPU infrastructure
- Built-in monitoring and logging
- Blue/green deployments with traffic splitting
- Integration with EcoSupplyAI's Azure ecosystem

```
EcoSupplyAI App --> Azure API Management --> Azure ML Managed Endpoint
                                                  |
                                          Fine-Tuned Model (GPU)
```

In [ ]:
# ============================================================
# Azure ML Deployment Pipeline (all commented - requires Azure setup)
# Prerequisites: pip install azure-ai-ml azure-identity
# ============================================================

# --- Step 1: Connect to Azure ML Workspace ---
# from azure.ai.ml import MLClient
# from azure.identity import DefaultAzureCredential
# from azure.ai.ml.entities import (
#     Model, ManagedOnlineEndpoint, ManagedOnlineDeployment, Environment,
# )
#
# credential = DefaultAzureCredential()
# ml_client = MLClient(
#     credential=credential,
#     subscription_id="<your-subscription-id>",
#     resource_group_name="ecosupplyai-rg",
#     workspace_name="ecosupplyai-ml-workspace",
# )

# --- Step 2: Register Model ---
# model = Model(
#     path=MERGED_DIR,
#     name="ecosupplyai-sustainability-llm",
#     description="Fine-tuned TinyLlama for supply chain sustainability Q&A",
#     type="custom_model",
#     tags={"framework": "transformers", "domain": "supply-chain-sustainability"},
# )
# registered_model = ml_client.models.create_or_update(model)

# --- Step 3: Create Endpoint ---
# endpoint = ManagedOnlineEndpoint(
#     name="ecosupplyai-llm-endpoint",
#     description="EcoSupplyAI Sustainability LLM Inference Endpoint",
#     auth_mode="key",
# )
# ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# --- Step 4: Create Environment ---
# env = Environment(
#     name="ecosupplyai-inference-env",
#     image="mcr.microsoft.com/azureml/curated/acft-hf-textgen-gpu:latest",
#     conda_file={"dependencies": ["python=3.10", "pip",
#         {"pip": ["torch>=2.0", "transformers>=4.36", "accelerate>=0.25",
#                  "peft>=0.7", "azureml-inference-server-http"]}]},
# )

# --- Step 5: Deploy ---
# deployment = ManagedOnlineDeployment(
#     name="ecosupplyai-llm-v1",
#     endpoint_name="ecosupplyai-llm-endpoint",
#     model=registered_model,
#     environment=env,
#     instance_type="Standard_NC4as_T4_v3",
#     instance_count=1,
# )
# ml_client.online_deployments.begin_create_or_update(deployment).result()
# endpoint.traffic = {"ecosupplyai-llm-v1": 100}
# ml_client.online_endpoints.begin_create_or_update(endpoint).result()

# --- Step 6: Test Endpoint ---
# test_request = {"input_data": {
#     "instruction": "What is the ESG score for a supplier with 500 tCO2e emissions?",
#     "input": "",
#     "parameters": {"max_new_tokens": 512, "temperature": 0.7},
# }}
# response = ml_client.online_endpoints.invoke(
#     endpoint_name="ecosupplyai-llm-endpoint",
#     request_file=json.dumps(test_request),
# )

print("Azure ML deployment code ready (commented out).")
print("Prerequisites: Azure subscription, azure-ai-ml package, completed training.")

## Summary & Next Steps

### What We Built

A complete fine-tuning pipeline for EcoSupplyAI covering:
1. **Dataset** - 22 instruction-tuning examples across 5 sustainability domains
2. **Preprocessing** - Alpaca-format prompt templating and tokenization
3. **LoRA fine-tuning** - Parameter-efficient adaptation (~1-2% of weights)
4. **Evaluation** - Perplexity comparison and qualitative assessment
5. **Export** - LoRA adapter saving, weight merging, model card
6. **Deployment** - Azure ML managed endpoint configuration

### Next Steps

| Priority | Task | Description |
|----------|------|-------------|
| P0 | Expand training data | 500-2000 examples for production quality |
| P0 | GPU training run | Execute on Azure ML Compute or equivalent |
| P1 | Human evaluation | Domain expert review of outputs |
| P1 | RAG integration | Combine with retrieval-augmented generation |
| P1 | Guardrails | Output validation for regulatory accuracy |
| P2 | Multi-adapter | Separate LoRA adapters per domain |
| P2 | Quantization | GPTQ/AWQ for faster inference |
| P2 | Continuous learning | Periodic re-training pipeline |
| P3 | Benchmark suite | Automated evaluation benchmarks |
| P3 | A/B testing | Fine-tuned vs API-based LLM in production |